# 淘宝用户行为分群与用户价值分析

本 Notebook 基于淘宝 `UserBehavior.csv` 全量数据，使用 `chunksize` 分块读取方式构建用户级行为特征，并通过规则法完成用户分层。

本阶段只做用户运营视角的行为分群和价值分析，不使用复杂机器学习模型。

## 一、分析目标

从电商平台用户运营视角，本分析重点回答以下问题：

1. 哪些用户是高价值购买用户？
2. 哪些用户浏览很多但购买少，存在转化潜力？
3. 哪些用户加购或收藏很多但没有购买，可能处于犹豫状态？
4. 哪些用户活跃度低，可能需要召回？
5. 不同用户群体应该采取什么运营策略？

## 二、导入库

导入数据处理、可视化和字典累计统计需要用到的工具包。

In [ ]:
# 导入数据处理工具包
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import defaultdict

# 导入 Markdown 展示工具，用于展示动态业务解读
from IPython.display import Markdown, display

# 设置中文字体，避免中文图表乱码
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# 设置 seaborn 图表风格
sns.set_theme(style="whitegrid", font="SimHei")

## 三、设置路径和基础参数

原始数据来自 E 盘项目目录，分群明细表和分群汇总表统一保存到 `data/processed/`。

In [ ]:
# 原始数据路径
raw_path = r"E:\ecommerce-user-growth-analysis\data\raw\UserBehavior.csv"

# 用户分群明细表输出路径
user_output_path = r"E:\ecommerce-user-growth-analysis\data\processed\user_segmentation.csv"

# 用户分群汇总表输出路径
summary_output_path = r"E:\ecommerce-user-growth-analysis\data\processed\user_segment_summary.csv"

# CSV 文件没有表头，因此手动指定列名
col_names = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]

# 分块大小：每次读取 100 万行，避免一次性读取 3.67GB 全量数据
chunksize = 1000000

# 输出目录不存在时自动创建
processed_dir = os.path.dirname(user_output_path)
os.makedirs(processed_dir, exist_ok=True)

# 行为类型顺序，用于后续统计和图表展示
behavior_order = ["pv", "fav", "cart", "buy"]

# 分群展示顺序，保证汇总表和图表符合业务优先级
segment_order = [
    "高价值用户",
    "普通购买用户",
    "高潜力用户",
    "犹豫收藏用户",
    "高浏览低转化用户",
    "低活跃用户",
    "其他用户",
]

print("原始数据路径：", raw_path)
print("用户分群明细输出路径：", user_output_path)
print("用户分群汇总输出路径：", summary_output_path)

## 四、分块读取全量数据，构建用户级特征

这一部分不会一次性读取全量数据。每个 chunk 只保留当前分块数据，并把统计结果累计到字典或集合中。

需要累计的用户特征包括行为次数、活跃日期、首次/最后活跃日期、互动商品数和互动类目数。

In [ ]:
# 如果原始数据不存在，提前报错，方便定位问题
if not os.path.exists(raw_path):
    raise FileNotFoundError(f"找不到原始数据文件：{raw_path}")

# 每个用户的总行为次数
user_total_count = defaultdict(int)

# 每个用户的四类行为次数
user_behavior_counts = defaultdict(lambda: defaultdict(int))

# 每个用户的活跃日期集合，用于计算 active_days
user_active_dates = defaultdict(set)

# 每个用户的首次活跃日期
user_first_date = {}

# 每个用户的最后活跃日期
user_last_date = {}

# 每个用户互动过的独立商品集合
user_unique_items = defaultdict(set)

# 每个用户互动过的独立类目集合
user_unique_categories = defaultdict(set)

# 记录处理进度
total_rows = 0
valid_rows = 0

# 使用 chunksize 分块读取全量数据
reader = pd.read_csv(
    raw_path,
    names=col_names,
    header=None,
    chunksize=chunksize,
)

# 逐个分块处理数据
for chunk_no, chunk in enumerate(reader, start=1):
    # 累计读取行数
    total_rows += len(chunk)

    # 将 timestamp 转换为 datetime，无法转换的值会变成 NaT
    chunk["datetime"] = pd.to_datetime(chunk["timestamp"], unit="s", errors="coerce")

    # 删除 datetime 为空的数据，避免错误时间影响活跃日期统计
    chunk = chunk.dropna(subset=["datetime"]).copy()

    # 从 datetime 中提取自然日期
    chunk["date"] = chunk["datetime"].dt.date

    # 累计有效行数
    valid_rows += len(chunk)

    # 统计每个用户的总行为次数
    total_counts = chunk.groupby("user_id").size()
    for user_id, count in total_counts.items():
        user_total_count[user_id] += int(count)

    # 统计每个用户的 pv、fav、cart、buy 次数
    behavior_counts = chunk.groupby(["user_id", "behavior_type"]).size()
    for (user_id, behavior_type), count in behavior_counts.items():
        user_behavior_counts[user_id][behavior_type] += int(count)

    # 统计每个用户的活跃日期、首次活跃日期和最后活跃日期
    user_date_pairs = chunk[["user_id", "date"]].drop_duplicates()
    for user_id, dates in user_date_pairs.groupby("user_id")["date"]:
        # 当前用户在这个分块中的活跃日期集合
        date_set = set(dates.tolist())

        # 更新活跃日期集合
        user_active_dates[user_id].update(date_set)

        # 当前分块中该用户最早和最晚活跃日期
        min_date = min(date_set)
        max_date = max(date_set)

        # 更新首次活跃日期
        if user_id not in user_first_date or min_date < user_first_date[user_id]:
            user_first_date[user_id] = min_date

        # 更新最后活跃日期
        if user_id not in user_last_date or max_date > user_last_date[user_id]:
            user_last_date[user_id] = max_date

    # 统计每个用户互动过的独立商品数
    user_item_pairs = chunk[["user_id", "item_id"]].drop_duplicates()
    for user_id, item_ids in user_item_pairs.groupby("user_id")["item_id"]:
        user_unique_items[user_id].update(item_ids.tolist())

    # 统计每个用户互动过的独立类目数
    user_category_pairs = chunk[["user_id", "category_id"]].drop_duplicates()
    for user_id, category_ids in user_category_pairs.groupby("user_id")["category_id"]:
        user_unique_categories[user_id].update(category_ids.tolist())

    # 打印处理进度，方便观察长任务运行情况
    print(f"已处理第 {chunk_no} 个分块，累计读取 {total_rows:,} 行，有效时间行 {valid_rows:,} 行，累计用户 {len(user_total_count):,} 个")

print("\n全量数据分块读取完成")
print(f"累计读取行数：{total_rows:,}")
print(f"有效时间行数：{valid_rows:,}")
print(f"累计用户数：{len(user_total_count):,}")

## 五、生成用户特征表

把前面累计在字典和集合中的结果整理成用户级特征表。每一行代表一个用户。

In [ ]:
# 汇总所有用户 ID
all_user_ids = sorted(user_total_count.keys())

# 逐个用户整理特征
user_rows = []
for user_id in all_user_ids:
    user_rows.append(
        {
            "user_id": user_id,
            "total_behavior_count": int(user_total_count[user_id]),
            "pv_count": int(user_behavior_counts[user_id]["pv"]),
            "fav_count": int(user_behavior_counts[user_id]["fav"]),
            "cart_count": int(user_behavior_counts[user_id]["cart"]),
            "buy_count": int(user_behavior_counts[user_id]["buy"]),
            "active_days": int(len(user_active_dates[user_id])),
            "first_active_date": user_first_date.get(user_id),
            "last_active_date": user_last_date.get(user_id),
            "unique_items": int(len(user_unique_items[user_id])),
            "unique_categories": int(len(user_unique_categories[user_id])),
        }
    )

# 生成用户特征 DataFrame
user_features = pd.DataFrame(user_rows)

# 日期字段转换为 pandas 日期类型，便于保存和查看
user_features["first_active_date"] = pd.to_datetime(user_features["first_active_date"])
user_features["last_active_date"] = pd.to_datetime(user_features["last_active_date"])

# 展示用户特征表前几行
display(user_features.head())
print(f"用户特征表行数：{len(user_features):,}")

## 六、新增衍生指标

衍生指标用于解释用户是否购买、是否有加购或收藏行为，以及用户从兴趣行为到购买之间的潜在转化空间。

In [ ]:
# 是否购买过：购买次数大于 0 记为 1，否则记为 0
user_features["purchase_flag"] = (user_features["buy_count"] > 0).astype(int)

# 是否加购过：加购次数大于 0 记为 1，否则记为 0
user_features["cart_flag"] = (user_features["cart_count"] > 0).astype(int)

# 是否收藏过：收藏次数大于 0 记为 1，否则记为 0
user_features["fav_flag"] = (user_features["fav_count"] > 0).astype(int)

# 行为深度：收藏、加购、购买属于比浏览更深的行为
user_features["behavior_depth"] = user_features["fav_count"] + user_features["cart_count"] + user_features["buy_count"]

# 购买率：购买次数 / 总行为次数，衡量用户行为中购买行为占比
user_features["purchase_rate"] = np.where(
    user_features["total_behavior_count"] > 0,
    user_features["buy_count"] / user_features["total_behavior_count"],
    0,
)

# 加购到购买潜力：加购次数多但购买次数少，可能说明转化空间更大
user_features["cart_to_buy_potential"] = user_features["cart_count"] - user_features["buy_count"]

# 收藏到购买潜力：收藏次数多但购买次数少，可能说明用户仍在比较或犹豫
user_features["fav_to_buy_potential"] = user_features["fav_count"] - user_features["buy_count"]

# 展示新增指标后的结果
display(user_features.head())

## 七、用户分层规则

这里使用规则法完成用户分群，不使用复杂机器学习。规则按业务优先级依次判断，一个用户只归入一个群体。

In [ ]:
# 计算 pv_count 的 75 分位数，用于识别高浏览低转化用户
pv_75 = user_features["pv_count"].quantile(0.75)

# 计算 total_behavior_count 的 25 分位数，用于识别低活跃用户
behavior_25 = user_features["total_behavior_count"].quantile(0.25)

# 定义用户分层函数，按题目给定顺序依次判断
def assign_user_segment(row):
    """根据规则给单个用户打上用户分群标签。"""
    # 1. 高价值用户：购买次数不少于 2，且活跃天数不少于 2
    if row["buy_count"] >= 2 and row["active_days"] >= 2:
        return "高价值用户"

    # 2. 普通购买用户：购买过，但不满足高价值用户条件
    if row["buy_count"] >= 1:
        return "普通购买用户"

    # 3. 高潜力用户：没有购买，但加购次数不少于 2
    if row["buy_count"] == 0 and row["cart_count"] >= 2:
        return "高潜力用户"

    # 4. 犹豫收藏用户：没有购买，但收藏次数不少于 2
    if row["buy_count"] == 0 and row["fav_count"] >= 2:
        return "犹豫收藏用户"

    # 5. 高浏览低转化用户：没有购买，且浏览次数达到用户浏览次数的 75 分位数
    if row["buy_count"] == 0 and row["pv_count"] >= pv_75:
        return "高浏览低转化用户"

    # 6. 低活跃用户：总行为次数不高于用户总行为次数的 25 分位数
    if row["total_behavior_count"] <= behavior_25:
        return "低活跃用户"

    # 7. 其他用户：不满足以上规则
    return "其他用户"


# 应用分层规则，生成 user_segment 字段
user_features["user_segment"] = user_features.apply(assign_user_segment, axis=1)

# 查看不同用户群体数量
display(user_features["user_segment"].value_counts().reindex(segment_order).dropna())
print(f"pv_count 75 分位数：{pv_75:.2f}")
print(f"total_behavior_count 25 分位数：{behavior_25:.2f}")

## 八、用户分群汇总

按用户群体统计用户数、用户占比和平均行为特征，便于比较不同群体的用户价值与运营重点。

In [ ]:
# 用户总数，用于计算用户占比
total_users = len(user_features)

# 按用户群体聚合核心指标
user_segment_summary = (
    user_features
    .groupby("user_segment")
    .agg(
        用户数=("user_id", "count"),
        平均总行为数=("total_behavior_count", "mean"),
        平均浏览数=("pv_count", "mean"),
        平均收藏数=("fav_count", "mean"),
        平均加购数=("cart_count", "mean"),
        平均购买数=("buy_count", "mean"),
        平均活跃天数=("active_days", "mean"),
        平均互动商品数=("unique_items", "mean"),
        平均互动类目数=("unique_categories", "mean"),
    )
    .reset_index()
    .rename(columns={"user_segment": "用户群体"})
)

# 计算用户占比
user_segment_summary["用户占比"] = user_segment_summary["用户数"] / total_users

# 按业务顺序排序
user_segment_summary["用户群体"] = pd.Categorical(
    user_segment_summary["用户群体"],
    categories=segment_order,
    ordered=True,
)
user_segment_summary = user_segment_summary.sort_values("用户群体").reset_index(drop=True)

# 调整列顺序，保证用户数和用户占比靠前
summary_columns = [
    "用户群体",
    "用户数",
    "用户占比",
    "平均总行为数",
    "平均浏览数",
    "平均收藏数",
    "平均加购数",
    "平均购买数",
    "平均活跃天数",
    "平均互动商品数",
    "平均互动类目数",
]
user_segment_summary = user_segment_summary[summary_columns]

# 展示汇总结果
display(user_segment_summary)

## 九、保存结果

保存用户分群明细表和用户分群汇总表，供后续运营分析或 BI 展示使用。

In [ ]:
# 保存用户分群明细表
user_features.to_csv(user_output_path, index=False, encoding="utf-8-sig")

# 保存用户分群汇总表
user_segment_summary.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

print("结果文件已保存：")
print(user_output_path)
print(summary_output_path)

## 十、可视化

通过图表观察不同用户群体的规模、占比、购买价值、活跃水平和行为结构差异。

In [ ]:
# 为可视化准备一个普通字符串类型的汇总表，避免分类类型在部分图表中显示异常
plot_summary = user_segment_summary.copy()
plot_summary["用户群体"] = plot_summary["用户群体"].astype(str)

# 1. 不同用户群体用户数量柱状图
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_summary, x="用户群体", y="用户数", color="#2F6B9A")
plt.title("不同用户群体用户数量")
plt.xlabel("用户群体")
plt.ylabel("用户数")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 2. 不同用户群体用户占比饼图
plt.figure(figsize=(8, 8))
plt.pie(
    plot_summary["用户数"],
    labels=plot_summary["用户群体"],
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
)
plt.title("不同用户群体用户占比")
plt.tight_layout()
plt.show()

In [ ]:
# 3. 不同用户群体平均购买次数柱状图
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_summary, x="用户群体", y="平均购买数", color="#B85C38")
plt.title("不同用户群体平均购买次数")
plt.xlabel("用户群体")
plt.ylabel("平均购买次数")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 4. 不同用户群体平均活跃天数柱状图
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_summary, x="用户群体", y="平均活跃天数", color="#5F8D4E")
plt.title("不同用户群体平均活跃天数")
plt.xlabel("用户群体")
plt.ylabel("平均活跃天数")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 5. 不同用户群体平均浏览、收藏、加购、购买行为对比图
behavior_compare = plot_summary.melt(
    id_vars="用户群体",
    value_vars=["平均浏览数", "平均收藏数", "平均加购数", "平均购买数"],
    var_name="行为指标",
    value_name="平均次数",
)

plt.figure(figsize=(13, 6))
sns.barplot(data=behavior_compare, x="用户群体", y="平均次数", hue="行为指标")
plt.title("不同用户群体平均行为次数对比")
plt.xlabel("用户群体")
plt.ylabel("平均次数")
plt.xticks(rotation=30, ha="right")
plt.legend(title="行为指标")
plt.tight_layout()
plt.show()

## 十一、业务解读

下面基于用户分群汇总结果自动生成业务解读，用于回答不同群体的规模、价值和运营动作。

In [ ]:
# 找出用户数量最多的群体
largest_segment_row = plot_summary.loc[plot_summary["用户数"].idxmax()]
largest_segment = largest_segment_row["用户群体"]
largest_segment_users = int(largest_segment_row["用户数"])
largest_segment_ratio = largest_segment_row["用户占比"]

# 找出平均购买次数最高的群体
highest_value_row = plot_summary.loc[plot_summary["平均购买数"].idxmax()]
highest_value_segment = highest_value_row["用户群体"]
highest_value_buy = highest_value_row["平均购买数"]

# 判断适合优惠券刺激的群体：优先选择高潜力用户，如果不存在则选择高浏览低转化用户
available_segments = set(plot_summary["用户群体"])
coupon_segment = "高潜力用户" if "高潜力用户" in available_segments else "高浏览低转化用户"

# 判断适合收藏/加购提醒的群体
reminder_segment = "犹豫收藏用户" if "犹豫收藏用户" in available_segments else "高潜力用户"

# 判断需要召回的群体
recall_segment = "低活跃用户" if "低活跃用户" in available_segments else "其他用户"

# 用 Markdown 展示业务解读
display(
    Markdown(
        f"""
### 1. 哪类用户数量最多？

用户数量最多的是 **{largest_segment}**，用户数为 **{largest_segment_users:,}**，占比约 **{largest_segment_ratio:.2%}**。该群体决定了平台用户结构的基本盘。

### 2. 哪类用户购买价值最高？

从平均购买次数看，购买价值最高的是 **{highest_value_segment}**，平均购买次数约为 **{highest_value_buy:.2f}**。该类用户应重点维护，避免流失，并持续促进复购。

### 3. 哪类用户最适合优惠券刺激？

**{coupon_segment}** 最适合优惠券刺激。该类用户通常已经表现出较强兴趣，例如加购多或浏览多，但尚未完成购买，优惠券、限时折扣和库存提醒更容易推动转化。

### 4. 哪类用户适合收藏/加购提醒？

**{reminder_segment}** 适合收藏/加购提醒。运营上可以结合降价提醒、评价内容、买家秀、相似商品推荐，降低用户决策成本。

### 5. 哪类用户需要召回？

**{recall_segment}** 需要重点召回。这类用户行为较少或近期互动不足，可通过召回券、个性化推荐、活动提醒提升回访。

### 6. 对不同用户群体的产品运营策略

高价值用户应重点维护和复购激励；普通购买用户应引导二次购买；高潜力用户适合限时优惠和加购提醒；犹豫收藏用户适合内容种草和相似商品推荐；高浏览低转化用户应优化推荐精准度、商品详情页和价格吸引力；低活跃用户应通过召回策略提升回访。
        """
    )
)

## 十二、运营策略建议

根据不同用户群体的典型特征，输出可直接用于运营动作设计的策略表。

In [ ]:
# 构建运营策略表
strategy_table = pd.DataFrame(
    [
        {
            "用户群体": "高价值用户",
            "用户特征": "购买次数多，活跃天数较高，复购潜力强",
            "业务问题": "需要重点维护，避免高价值用户流失",
            "运营策略": "推荐会员权益、专属优惠、复购激励、优先客服和新品尝鲜权益",
        },
        {
            "用户群体": "普通购买用户",
            "用户特征": "已经完成购买，但购买频次和活跃深度有限",
            "业务问题": "需要从首购转向复购，提高生命周期价值",
            "运营策略": "推送相关商品、满减券、组合推荐、订单后关怀和复购提醒",
        },
        {
            "用户群体": "高潜力用户",
            "用户特征": "已有多次加购行为但尚未购买",
            "业务问题": "临门一脚转化不足，可能受价格、库存或决策成本影响",
            "运营策略": "推送限时优惠、库存提醒、降价提醒、购物车优惠券和免邮门槛提示",
        },
        {
            "用户群体": "犹豫收藏用户",
            "用户特征": "收藏较多但尚未购买，兴趣明确但决策偏弱",
            "业务问题": "用户可能仍在比较商品，需要增强信任和购买理由",
            "运营策略": "推送商品评价、买家秀、相似商品推荐、降价提醒和收藏夹专题推荐",
        },
        {
            "用户群体": "高浏览低转化用户",
            "用户特征": "浏览次数高但没有购买",
            "业务问题": "可能存在推荐不准、详情页信息不足或价格吸引力不够的问题",
            "运营策略": "优化推荐精准度、详情页信息、价格表达、优惠露出和商品对比体验",
        },
        {
            "用户群体": "低活跃用户",
            "用户特征": "总行为次数低，互动深度弱",
            "业务问题": "用户回访动力不足，存在沉默或流失风险",
            "运营策略": "通过召回券、个性化推荐、活动提醒、新人任务和兴趣频道提升回访",
        },
        {
            "用户群体": "其他用户",
            "用户特征": "未满足重点分群规则，行为特征相对分散",
            "业务问题": "需要继续观察行为变化，识别后续转化或流失信号",
            "运营策略": "使用常规推荐、活动曝光、内容导购和基础权益维持活跃",
        },
    ]
)

# 展示策略表
display(strategy_table)